# Cloud Application Root Cause Analysis

> **Data note:** The original application-trace dataset is not included in this public repository. Place the authorized source file at `data/raw/Reddit.txt` and any generated intermediate CSV files in `data/processed/` before running the relevant cells. The original notebook was developed in a research environment and may require dependency/version adjustments; this repository does not claim one-click reproducibility without the private source data.

This notebook explores patterns associated with erroneous application traces, compares analytical approaches, and examines feature-importance and explainability techniques. Patterns associated with errors should not be interpreted as independently verified operational root causes.


# Intro

In [ ]:
import json
import pandas as pd
import numpy as np
import sys
from sklearn.model_selection import train_test_split

In [ ]:
from eli5.sklearn import PermutationImportance

In [ ]:
import shap
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
import xgboost
from sklearn.compose import ColumnTransformer
from sklearn.neural_network import MLPClassifier

In [ ]:
import lime 
from lime import lime_tabular

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import PrecisionRecallDisplay

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from numpy import loadtxt
from sklearn.inspection import permutation_importance

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Flatten
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from keras import regularizers
from keras.layers import Dense, Dropout

# Output

In [ ]:
with open('data/raw/Reddit.txt', encoding='utf-8') as f:
    data = f.read()

In [ ]:
data = data.split('data:')
data = data[1:]
number_of_traces = len(data)
number_of_traces

In [ ]:
traces = []
for j in np.arange(number_of_traces):
    traces.append(eval(data[j]))

In [ ]:
ll=[]
for j in np.arange(number_of_traces):
    ll.append(len(traces[j]['spans']))

In [ ]:
plt.figure(figsize = (12,5))
plt.scatter(np.arange(number_of_traces), ll)
#plt.title('The number of spans in different traces')

In [ ]:
traces[10]

In [ ]:
def reduce(function, iterable, initializer=None):
    it = iter(iterable)
    if initializer is None:
        try:
            initializer = next(it)
        except StopIteration:
            raise TypeError('reduce() of empty sequence with no initial value')
    accum_value = initializer
    for x in it:
        accum_value = function(accum_value, x)
    return accum_value

In [ ]:
output = np.zeros(number_of_traces)
for j in np.arange(number_of_traces):
    trace = traces[j]
    spans = trace['spans']
    number_of_spans = len(spans)
    for i in np.arange(number_of_spans):
        dicts = spans[i]['annotations']
        new_dicts = reduce(lambda d, src: d.update(src) or d, dicts, {})
        if 'error' in new_dicts.keys():
            output[j] = 1 
            break

In [ ]:
np.unique(output, return_counts = True)

# Datasets

In [ ]:
all_spans_name = []
numbers = []
for j in np.arange(number_of_traces):
    trace = traces[j]
    spans = trace['spans']
    number_of_spans = len(spans)
    numbers.append(number_of_spans)
    for i in np.arange(number_of_spans):
        name = spans[i]['name']
        all_spans_name.append(name) 
unique_spans_names = np.unique(all_spans_name)
len(unique_spans_names)

In [ ]:
np.set_printoptions(threshold=5)
unique_spans_names

In [ ]:
df_span_names = pd.DataFrame(index = np.arange(number_of_traces), columns = unique_spans_names)

In [ ]:
for j in np.arange(number_of_traces):
    trace = traces[j]
    spans = trace['spans']
    number_of_spans = len(spans)
    for i in np.arange(number_of_spans):
        name = spans[i]['name']
        df_span_names.loc[j,name] = 1

In [ ]:
df_span_names.shape

In [ ]:
total_number_of_values = number_of_traces * len(unique_spans_names)
np.sum(np.sum(df_span_names.isnull()))/total_number_of_values*100

In [ ]:
df_span_names_repetitions = pd.DataFrame(index = np.arange(number_of_traces), columns = unique_spans_names)

In [ ]:
for j in np.arange(number_of_traces):
    trace = traces[j]
    spans = trace['spans']
    number_of_spans = len(spans)
    name = []
    for i in np.arange(number_of_spans):
        name.append(spans[i]['name']) 
    span_name_counts = np.unique(name,return_counts = True) 
    for k in np.arange(len(span_name_counts[0])):
        df_span_names_repetitions.loc[j,span_name_counts[0][k]] = span_name_counts[1][k]

In [ ]:
df_span_names_repetitions.shape

In [ ]:
column_names = []
for j in np.arange(number_of_traces):
    length_of_trace = len(traces[j]['spans'])
    for i in np.arange(length_of_trace):
        name1 = traces[j]['spans'][i]['name'] + '_host'
        column_names.append(name1)
        name2 = traces[j]['spans'][i]['name'] + '_durationMs'
        column_names.append(name2)
        dicts = traces[j]['spans'][i]['annotations']
        new_dicts = reduce(lambda d, src: d.update(src) or d, dicts, {})
        del(new_dicts['spanId'])
        del(new_dicts['traceId'])
        if 'error' in new_dicts.keys(): 
            del(new_dicts['error'])
        if 'parent' in new_dicts.keys(): 
            del(new_dicts['parent'])
        if 'followsFrom' in new_dicts.keys(): 
            del(new_dicts['followsFrom'])
        annotation_names = list(new_dicts.keys())
        annotation_names_length = len(annotation_names)
        for s in np.arange(annotation_names_length):
            name = traces[j]['spans'][i]['name'] + '_annotations' + '_' + annotation_names[s]
            column_names.append(name)
unique_column_names = np.unique(column_names)

In [ ]:
len(unique_column_names)

In [ ]:
unique_column_names

In [ ]:
df_dimensions = pd.DataFrame(index = np.arange(number_of_traces), columns = unique_column_names)

In [ ]:
name_list = []
for j in np.arange(number_of_traces):
    length_of_trace = len(traces[j]['spans'])
    for i in np.arange(length_of_trace):
        name1 = traces[j]['spans'][i]['name'] + '_host'
        df_dimensions.loc[j,name1] = traces[j]['spans'][i]['host']
        name2 = traces[j]['spans'][i]['name'] + '_durationMs'
        df_dimensions.loc[j,name2] = traces[j]['spans'][i]['durationMs']
        dicts = traces[j]['spans'][i]['annotations']
        new_dicts = reduce(lambda d, src: d.update(src) or d, dicts, {})
        del(new_dicts['spanId'])
        del(new_dicts['traceId'])
        if 'error' in new_dicts.keys(): 
            del(new_dicts['error'])
        if 'parent' in new_dicts.keys(): 
            del(new_dicts['parent'])
        if 'followsFrom' in new_dicts.keys(): 
            del(new_dicts['followsFrom'])
        annotation_names = list(new_dicts.keys())
        for s in list(new_dicts.keys()):
            name = traces[j]['spans'][i]['name'] + '_annotations' + '_' + s 
            df_dimensions.loc[j,name] = new_dicts[s]

In [ ]:
df_dimensions.shape

In [ ]:
total_number_of_values = number_of_traces * (len(unique_spans_names) + len(unique_column_names))
np.sum(np.sum(df_dimensions.isnull()))/total_number_of_values*100

# Data Short

In [ ]:
dd = df_span_names.loc[output == 1,:]
isnull_index = np.sum(dd.isnull()) == dd.shape[0]
df_span_names_short = df_span_names.drop(dd.columns[isnull_index], axis = 1)
df_span_names_repetitions_short = df_span_names_repetitions.drop(dd.columns[isnull_index], axis = 1)

In [ ]:
df_span_names_short.shape

In [ ]:
df_span_names_repetitions_short.shape

In [ ]:
df_span_names_short['output'] = output
df_span_names_short.to_csv("data/processed/df_span_names_short.csv", index = False)

In [ ]:
df_span_names_repetitions_short['output'] = output
df_span_names_repetitions_short.to_csv("data/processed/df_span_names_repetitions_short.csv", 
                                       index = False)

In [ ]:
dd = df_dimensions.loc[output == 1,:]
isnull_index = np.sum(dd.isnull()) == dd.shape[0]
df_dimensions_short = df_dimensions.drop(dd.columns[isnull_index], axis = 1)

In [ ]:
df_dimensions_short.shape

In [ ]:
df_dimensions_short['output'] = output
df_dimensions_short.to_csv("data/processed/df_dimensions_short.csv", index = False)

# Data Download and Transformations

In [ ]:
df_span_names_short = pd.read_csv("data/processed/df_span_names_short.csv")
y = df_span_names_short['output'].astype('category')
X_df_span_names_short = df_span_names_short.drop(['output'], axis = 1).astype('category')

In [ ]:
np.sum(X_df_span_names_short.dtypes == 'category')

In [ ]:
df_span_names_repetitions_short = pd.read_csv("data/processed/df_span_names_repetitions_short.csv")
X_df_span_names_repetitions_short = df_span_names_repetitions_short.drop(['output'], axis = 1).astype('category')

In [ ]:
np.sum(X_df_span_names_repetitions_short.dtypes == 'category')

In [ ]:
df_dimensions_short = pd.read_csv("data/processed/df_dimensions_short.csv",
                                 low_memory = False)
X_df_dimensions_short = df_dimensions_short.drop(['output'], axis = 1)

In [ ]:
import re
index_num = []
index_cat = []
index_card = []
for j in X_df_dimensions_short.columns:
    result = re.search("durationMs", j)
    cardinality = X_df_dimensions_short[j].nunique()
    if (result != None):
        index_num.append(j)
    elif (cardinality > 20):
        index_card.append(j)
    else:
        index_cat.append(j)

In [ ]:
mm = OrdinalEncoder()
m1 = pd.DataFrame(mm.fit_transform(X_df_dimensions_short[index_cat]), columns = index_cat).astype('category')
m2 = pd.DataFrame(mm.fit_transform(X_df_dimensions_short[index_card]), columns = index_card).astype('float64')
m3 = X_df_dimensions_short[index_num].astype('float64')
df_dimensions_trans = pd.concat([m1,m2,m3], axis = 1)

In [ ]:
df_dimensions_trans_w_durations = pd.concat([m1,m2], axis = 1)

In [ ]:
X_df_span_names_repetitions_short.shape

In [ ]:
ind = X_df_span_names_repetitions_short.nunique() < 150
X_df_span_names_repetitions_short = X_df_span_names_repetitions_short.loc[:,ind]

In [ ]:
X_df_span_names_repetitions_short.shape

In [ ]:
df_full_short = pd.concat([X_df_span_names_short, df_dimensions_trans], axis = 1)
df_full_repetitions_short = pd.concat([X_df_span_names_repetitions_short, df_dimensions_trans], axis = 1)
df_full_short_w_durations = pd.concat([X_df_span_names_short, df_dimensions_trans_w_durations], axis = 1)
df_full_short_repetitions_w_durations = pd.concat([X_df_span_names_repetitions_short, df_dimensions_trans_w_durations], axis = 1)

In [ ]:
# for Weka
#df_full_short['output'] = output
#df_full_short.to_csv("C:\\Users\\apoghosyan\\Desktop\\Lilit Capstone\\df_full_short.csv", index = False)
#df_full_short.drop(['output'], axis = 1, inplace = True)

#df_full_repetitions_short['output'] = output
#df_full_repetitions_short.to_csv("C:\\Users\\apoghosyan\\Desktop\\Lilit Capstone\\df_full_repetitions_short.csv", index = False)
#df_full_repetitions_short.drop(['output'], axis = 1, inplace = True)

#df_full_short_w_durations['output'] = output
#df_full_short_w_durations.to_csv("C:\\Users\\apoghosyan\\Desktop\\Lilit Capstone\\df_full_short_w_durations.csv", index = False)
#df_full_short_w_durations.drop(['output'], axis = 1, inplace = True)

#df_full_short_repetitions_w_durations['output'] = output
#df_full_short_repetitions_w_durations.to_csv("C:\\Users\\apoghosyan\\Desktop\\Lilit Capstone\\df_full_short_repetitions_w_durations.csv", index = False)
#df_full_short_repetitions_w_durations.drop(['output'], axis = 1, inplace = True)

# XGBoost - Span Names

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_df_span_names_short, y, random_state = 0)

In [ ]:
model_xgb = XGBClassifier(tree_method = "gpu_hist", enable_categorical = True)
n_estimators_grid = np.arange(5, 200, 5)
learning_rate_grid = [0.00001, 0.0001, 0.001, 0.01, 0.1, 1]
model_optimal_xgb = GridSearchCV(model_xgb, 
        param_grid = {'n_estimators': n_estimators_grid,
                     'learning_rate': learning_rate_grid},
                     n_jobs = -1, verbose = 5, return_train_score = True)
model_optimal_xgb.fit(X_train, y_train)

In [ ]:
model_optimal_xgb.best_params_

In [ ]:
pred_xgb = model_optimal_xgb.predict(X_train)
print(classification_report(y_train, pred_xgb))

In [ ]:
pred_xgb = model_optimal_xgb.predict(X_test)
print(classification_report(y_test, pred_xgb))

In [ ]:
test = model_optimal_xgb.cv_results_['mean_test_score'].reshape(len(learning_rate_grid), len(n_estimators_grid))[5]
test_std = model_optimal_xgb.cv_results_['std_test_score'].reshape(len(learning_rate_grid), len(n_estimators_grid))[5]
train =  model_optimal_xgb.cv_results_['mean_train_score'].reshape(len(learning_rate_grid), len(n_estimators_grid))[5]
train_std =  model_optimal_xgb.cv_results_['std_train_score'].reshape(len(learning_rate_grid), len(n_estimators_grid))[5]
plt.figure(figsize = (7,7))
plt.plot(n_estimators_grid, test, label = "Test", c = 'red')
plt.fill_between(x = n_estimators_grid, y1 = test- test_std, y2 = test + test_std, color = "lightgrey")
plt.plot(n_estimators_grid, train, label = "Train")
plt.legend()
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('CV Accuracy')
plt.title('XGBoostClassifier')
plt.show()


In [ ]:
model_final_xgb = XGBClassifier(tree_method = "gpu_hist", enable_categorical = True, n_estimators = 60, learning_rate = 1)
model_final_xgb.fit(X_df_span_names_short, y)

In [ ]:
sorted(list(zip(X_df_span_names_short.columns, model_final_xgb.feature_importances_)), 
       key = lambda x: abs(x[1]), reverse = True)

In [ ]:
imp = sorted(list(zip(X_df_span_names_short.columns, model_final_xgb.feature_importances_)), 
       key = lambda x: abs(x[1]), reverse = True)
names = []
scores = []
for j in np.arange(5):
    names.append(imp[j][0])
    scores.append(imp[j][1])

In [ ]:
importances = scores
forest_importances = pd.Series(scores, index = names)

fig, ax = plt.subplots(figsize = (9,6))
forest_importances.plot.bar(ax = ax)
ax.set_title("Feature Importances using XGBoost")
ax.set_ylabel("Importance Scores")
fig.tight_layout()

In [ ]:
result = permutation_importance(model_final_xgb, X_df_span_names_short, y, n_repeats = 5, random_state = 42, n_jobs = -1)

forest_importances = pd.Series(result.importances_mean, index = X_df_span_names_short.columns)

In [ ]:
imp = sorted(list(zip(X_df_span_names_short.columns,result['importances_mean'])), key = lambda x: abs(x[1]), reverse = True)
names = []
scores = []
for j in np.arange(5):
    names.append(imp[j][0])
    scores.append(imp[j][1])

In [ ]:
importances = scores
forest_importances = pd.Series(scores, index = names)

fig, ax = plt.subplots(figsize=(9,6))
forest_importances.plot.bar(ax=ax)
ax.set_title("Feature Importances via Permutations (XGBoostClassifier)")
ax.set_ylabel("Importance Scores")
fig.tight_layout()

In [ ]:
model = HistGradientBoostingClassifier(categorical_features = [True]*X_df_span_names_short.shape[1])
learning_rate_grid = [0.0001, 0.001, 0.01, 0.1, 0.5, 1]
model_optimal = GridSearchCV(model, 
        param_grid = {'learning_rate': learning_rate_grid},
                     n_jobs = -1, verbose = 5, return_train_score = True)
model_optimal.fit(X_train, y_train)

In [ ]:
model_optimal.best_params_

In [ ]:
pred_xgb = model_optimal.predict(X_test)
print(classification_report(y_test, pred_xgb))

In [ ]:
model_final = HistGradientBoostingClassifier(learning_rate = 0.01, categorical_features = [True]*X_df_span_names_short.shape[1])
model_final.fit(X_df_span_names_short, y)

In [ ]:
result = permutation_importance( model_final, X_df_span_names_short, y, n_repeats = 5, 
    random_state = 42, n_jobs = -1)

forest_importances = pd.Series(result.importances_mean, index = X_df_span_names_short.columns)

In [ ]:
imp = sorted(list(zip(X_df_span_names_short.columns,result['importances_mean'])), 
             key = lambda x: abs(x[1]), reverse = True)
names = []
scores = []
for j in np.arange(5):
    names.append(imp[j][0])
    scores.append(imp[j][1])

In [ ]:
importances = scores
forest_importances = pd.Series(scores, index = names)

fig, ax = plt.subplots(figsize=(9,6))
forest_importances.plot.bar(ax=ax)
ax.set_title("Feature Importances via Permutations (HistGradientBoostingClassifier)")
ax.set_ylabel("Importance Scores")
fig.tight_layout()

In [ ]:
dd = X_df_span_names_short.loc[y == 1,:]
y_short = y[y==1]

In [ ]:
sv = shap.TreeExplainer(model_final,  enable_categorical = True)
shap_values = sv.shap_values(dd.values, y_short.values)
shap.summary_plot(shap_values, dd.values, feature_names = dd.columns)

# XGBoost - Span Names  with Repetitions

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_df_span_names_repetitions_short, y)

In [ ]:
model = HistGradientBoostingClassifier( )
learning_rate_grid = [0.0001, 0.001, 0.01, 0.1, 0.5, 1]
model_optimal = GridSearchCV(model, 
        param_grid = {'learning_rate': learning_rate_grid},
                     n_jobs = -1, verbose = 5, return_train_score = True)
model_optimal.fit(X_train, y_train)

In [ ]:
model_optimal.best_params_

In [ ]:
pred_xgb = model_optimal.predict(X_test)
print(classification_report(y_test, pred_xgb))

In [ ]:
model_final = HistGradientBoostingClassifier(learning_rate = 1)
model_final.fit(X_df_span_names_repetitions_short, y)

In [ ]:
result = permutation_importance(
   model_final, X_df_span_names_repetitions_short, y, n_repeats = 5, 
    random_state = 42, n_jobs = -1)

forest_importances = pd.Series(result.importances_mean, index = X_df_span_names_repetitions_short.columns)

In [ ]:
imp = sorted(list(zip(X_df_span_names_repetitions_short.columns,result['importances_mean'])), 
             key = lambda x: abs(x[1]), reverse = True)
names = []
scores = []
for j in np.arange(5):
    names.append(imp[j][0])
    scores.append(imp[j][1])

In [ ]:
importances = scores
forest_importances = pd.Series(scores, index = names)

fig, ax = plt.subplots(figsize=(9,6))
forest_importances.plot.bar(ax=ax)
ax.set_title("Feature Importances using Permutations (HistGradientBoostingClassifier)")
ax.set_ylabel("Importance Scores")
fig.tight_layout()

In [ ]:
dd = X_df_span_names_repetitions_short.loc[y == 1,:]
y_short = y[y==1]

In [ ]:
sv = shap.TreeExplainer(model_final,  enable_categorical = True)
shap_values = sv.shap_values(dd,y_short)

In [ ]:
shap.summary_plot(shap_values, dd.values, feature_names = X_df_span_names_repetitions_short.columns)

# Full 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_full_short, y)

In [ ]:
model = HistGradientBoostingClassifier(categorical_features = [True]*199 + [True]*len(index_cat) + [False]*len(index_card) + [False]*len(index_num))
learning_rate_grid = [0.0001, 0.001, 0.01, 0.1, 0.5, 1]
model_optimal = GridSearchCV(model, 
        param_grid = {'learning_rate': learning_rate_grid},
                     n_jobs = -1, verbose = 5, return_train_score = True)
model_optimal.fit(X_train, y_train)

In [ ]:
model_optimal.best_params_

In [ ]:
pred_xgb = model_optimal.predict(X_test)
print(classification_report(y_test, pred_xgb))

In [ ]:
model_final = HistGradientBoostingClassifier(learning_rate = 0.01, 
            categorical_features = [True]*199 + [True]*len(index_cat) + [False]*len(index_card) + [False]*len(index_num))
model_final.fit(df_full_short, y)

In [ ]:
result = permutation_importance(
   model_final, df_full_short, y, n_repeats = 5, 
    random_state = 42, n_jobs = -1)

forest_importances = pd.Series(result.importances_mean, index = df_full_short.columns)

In [ ]:
imp = sorted(list(zip(df_full_short.columns, result['importances_mean'])), key = lambda x: abs(x[1]), reverse = True)
names = []
scores = []
for j in np.arange(5):
    names.append(imp[j][0])
    scores.append(imp[j][1])

In [ ]:
importances = scores
forest_importances = pd.Series(scores, index = names)

fig, ax = plt.subplots(figsize=(9,8))
forest_importances.plot.bar(ax=ax)
ax.set_title("Feature Importances using Permutations (HistGradientBoostingClassifier)")
ax.set_ylabel("Importance Scores")
fig.tight_layout()

In [ ]:
dd = df_full_short.loc[y == 1,:]
y_short = y[y==1]

In [ ]:
sv = shap.TreeExplainer(model_final,  enable_categorical = True)
shap_values = sv.shap_values(dd,y_short)

In [ ]:
shap.summary_plot(shap_values, dd.values,  feature_names = dd.columns)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_full_short_w_durations, y)

In [ ]:
model = HistGradientBoostingClassifier(categorical_features = [True]*199 + [True]*len(index_cat) + [False]*len(index_card) )
learning_rate_grid = [0.0001, 0.001, 0.01, 0.1, 0.5, 1]
model_optimal = GridSearchCV(model, 
        param_grid = {'learning_rate': learning_rate_grid},
                     n_jobs = -1, verbose = 5, return_train_score = True)
model_optimal.fit(X_train, y_train)

In [ ]:
model_optimal.best_params_

In [ ]:
pred_xgb = model_optimal.predict(X_test)
print(classification_report(y_test, pred_xgb))

In [ ]:
model_final = HistGradientBoostingClassifier(learning_rate = 0.01, 
            categorical_features = [True]*199 + [True]*len(index_cat) + [False]*len(index_card))
model_final.fit(df_full_short_w_durations, y)

In [ ]:
result = permutation_importance(
   model_final, df_full_short_w_durations, y, n_repeats = 5, 
    random_state = 42, n_jobs = -1)

forest_importances = pd.Series(result.importances_mean, index = df_full_short_w_durations.columns)

In [ ]:
imp = sorted(list(zip(df_full_short_w_durations.columns, result['importances_mean'])), 
             key = lambda x: abs(x[1]), reverse = True)
names = []
scores = []
for j in np.arange(5):
    names.append(imp[j][0])
    scores.append(imp[j][1])

In [ ]:
importances = scores
forest_importances = pd.Series(scores, index = names)

fig, ax = plt.subplots(figsize=(9,8))
forest_importances.plot.bar(ax=ax)
ax.set_title("Feature Importances using Permutations (HistGradientBoostingClassifier)")
ax.set_ylabel("Importance Scores")
fig.tight_layout()

In [ ]:
dd = df_full_short_w_durations.loc[y == 1,:]
y_short = y[y==1]

In [ ]:
sv = shap.TreeExplainer(model_final,  enable_categorical = True)
shap_values = sv.shap_values(dd,y_short)

In [ ]:
shap.summary_plot(shap_values, dd.values,  feature_names = dd.columns)

# Full  with  Repetitions

In [ ]:
df_full_repetitions_short.shape

In [ ]:
plt.plot(X_df_span_names_repetitions_short.nunique().values)
plt.title("Number of Repetitions")
plt.xlabel("Traces")
plt.ylabel("Number of Spans")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_full_repetitions_short, y)

In [ ]:
model = HistGradientBoostingClassifier(categorical_features = [True]*197 + [True]*len(index_cat) + [False]*len(index_card) + [False]*len(index_num))
learning_rate_grid = [0.0001, 0.001, 0.01, 0.1, 0.5, 1]
model_optimal = GridSearchCV(model, 
        param_grid = {'learning_rate': learning_rate_grid},
                     n_jobs = -1, verbose = 5, return_train_score = True)
model_optimal.fit(X_train, y_train)

In [ ]:
model_optimal.best_params_

In [ ]:
pred_xgb = model_optimal.predict(X_test)
print(classification_report(y_test, pred_xgb))

In [ ]:
model_final = HistGradientBoostingClassifier(learning_rate = 0.1, 
            categorical_features = [True]*197 + [True]*len(index_cat) + [False]*len(index_card) + [False]*len(index_num))
model_final.fit(df_full_repetitions_short, y)

In [ ]:
result = permutation_importance(model_final, df_full_repetitions_short, y, n_repeats = 5, 
    random_state = 42, n_jobs = -1)

forest_importances = pd.Series(result.importances_mean, index = df_full_repetitions_short.columns)

imp = sorted(list(zip(df_full_repetitions_short.columns, result['importances_mean'])), 
             key = lambda x: abs(x[1]), reverse = True)
names = []
scores = []
for j in np.arange(5):
    names.append(imp[j][0])
    scores.append(imp[j][1])

importances = scores
forest_importances = pd.Series(scores, index = names)

fig, ax = plt.subplots(figsize=(9,8))
forest_importances.plot.bar(ax=ax)
ax.set_title("Feature Importances using Permutations (HistGradientBoostingClassifier)")
ax.set_ylabel("Importance Scores")
fig.tight_layout()

In [ ]:
dd = df_full_repetitions_short.loc[y == 1,: ]
y_short = y[y==1]

sv = shap.TreeExplainer(model_final,  enable_categorical = True)
shap_values = sv.shap_values(dd, y_short)

shap.summary_plot(shap_values, dd.values,  feature_names = dd.columns)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_full_short_repetitions_w_durations, y)

In [ ]:
model = HistGradientBoostingClassifier(categorical_features = [True]*197 + [True]*len(index_cat) + [False]*len(index_card) )
learning_rate_grid = [0.0001, 0.001, 0.01, 0.1, 0.5, 1]
model_optimal = GridSearchCV(model, 
        param_grid = {'learning_rate': learning_rate_grid},
                     n_jobs = -1, verbose = 5, return_train_score = True)
model_optimal.fit(X_train, y_train)

In [ ]:
model_optimal.best_params_

In [ ]:
pred_xgb = model_optimal.predict(X_test)
print(classification_report(y_test, pred_xgb))

In [ ]:
model_final = HistGradientBoostingClassifier(learning_rate = 0.1, 
            categorical_features = [True]*197 + [True]*len(index_cat) + [False]*len(index_card))
model_final.fit(df_full_short_repetitions_w_durations, y)

In [ ]:
result = permutation_importance(
   model_final, df_full_short_repetitions_w_durations, y, n_repeats = 5, 
    random_state = 42, n_jobs = -1)

forest_importances = pd.Series(result.importances_mean, index = df_full_short_repetitions_w_durations.columns)

imp = sorted(list(zip(df_full_short_repetitions_w_durations.columns, result['importances_mean'])), key = lambda x: abs(x[1]), reverse = True)
names = []
scores = []
for j in np.arange(5):
    names.append(imp[j][0])
    scores.append(imp[j][1])

importances = scores
forest_importances = pd.Series(scores, index = names)

fig, ax = plt.subplots(figsize=(9,8))
forest_importances.plot.bar(ax=ax)
ax.set_title("Feature Importances using Permutations (HistGradientBoostingClassifier)")
ax.set_ylabel("Importance Scores")
fig.tight_layout()

In [ ]:
dd = df_full_short_repetitions_w_durations.loc[y == 1,:]
y_short = y[y==1]

sv = shap.TreeExplainer(model_final,  enable_categorical = True)
shap_values = sv.shap_values(dd,y_short)

shap.summary_plot(shap_values, dd.values,  feature_names = dd.columns)

# Visualization

In [ ]:
from sklearn.metrics import jaccard_score
from sklearn.metrics.pairwise import pairwise_distances

In [ ]:
from time import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
from sklearn import manifold
from sklearn.utils import check_random_state

# Unused but required import for doing 3d projections with matplotlib < 3.2
import mpl_toolkits.mplot3d  # noqa: F401
import warnings

In [ ]:
from sklearn.preprocessing import LabelBinarizer
from sklearn.cluster import AgglomerativeClustering
from matplotlib.colors import ListedColormap
from sklearn.metrics import pairwise_distances

In [ ]:
X = df_span_names.drop(['output'], axis = 1)
y = df_span_names['output']
mm = SimpleImputer(strategy = 'constant', fill_value = 0)
X_0 = mm.fit_transform(X)
X_0 = pd.DataFrame(X_0, columns =  X.columns)

In [ ]:
from scipy.spatial.distance import cdist

In [ ]:
jac_sim = cdist(X_0, X_0, metric = 'jaccard')

In [ ]:
tsne = manifold.TSNE(n_components = 2,  random_state = 0, learning_rate = "auto", metric = 'precomputed')
trans_data = tsne.fit_transform(jac_sim).T

plt.figure(figsize=(15,8))
plt.scatter(trans_data[0], trans_data[1], c = y.ravel(), cmap =  ListedColormap(['black','red']))
plt.title('Traces')

In [ ]:
tsne = manifold.TSNE(n_components = 3,  random_state = 0, learning_rate = "auto", metric = 'precomputed')
trans_data = tsne.fit_transform(jac_sim).T

fig = plt.figure(figsize = (12,12))
ax = plt.axes(projection = '3d')

ax.scatter(trans_data[0], trans_data[1], trans_data[2], c = y.ravel(), 
           cmap =  ListedColormap(['black','red']), s = 5)
ax.text3D(0, 0, 35, "Real Problems?", color = 'blue', weight = 'bold', fontsize = 'xx-large', fontstyle = 'italic')

a = Arrow3D([6, 0.5], [6, 1.5], [33, 7], mutation_scale = 20,
            lw = 2, arrowstyle = "-|>", color = "blue")
b = Arrow3D([6, 1], [6, 2], [33, -12], mutation_scale = 20,
            lw = 2, arrowstyle = "-|>", color = "blue")
c = Arrow3D([6, -11], [6, 1.5], [33, -8], mutation_scale = 20,
            lw = 2, arrowstyle = "-|>", color = "blue")
ax.add_artist(a)
ax.add_artist(b)
ax.add_artist(c)

plt.title('Traces')

In [ ]:
y = df_full['output']
df_full = df_full.drop(['output'], axis = 1)
import re
index_num = []
index_cat = []
index_del = []
index = []
for j in df_full.columns:
    result = re.search("durationMs", j)
    cardinality = len(df_full[j].unique())
    if (result != None):
        index_num.append(j)
        index.append(False)
    elif (cardinality > 20):
        index.append(False)
        index_del.append(j)
    else:
        index.append(True)
        index_cat.append(j)

In [ ]:
df_full = df_full.drop(index_del, axis = 1)
df_full = df_full.drop(index_num, axis = 1)

In [ ]:
trans = OrdinalEncoder()
df_full_trans = pd.DataFrame(trans.fit_transform(df_full), columns = df_full.columns)
mm = SimpleImputer(strategy = 'constant', fill_value = 0)
X_0 = mm.fit_transform(df_full_trans)

In [ ]:
X_0[X_0 > 0] = 1

In [ ]:
jac_sim = cdist(X_0, X_0, metric = 'jaccard')

In [ ]:
tsne = manifold.TSNE(n_components = 2,  random_state = 0, learning_rate = "auto", metric = 'precomputed')
trans_data = tsne.fit_transform(jac_sim).T

plt.figure(figsize=(15,8))
plt.scatter(trans_data[0], trans_data[1], c = y.ravel(), cmap =  ListedColormap(['black','red']))
plt.title('Traces')

In [ ]:
from mpl_toolkits import mplot3d

In [ ]:
tsne = manifold.TSNE(n_components = 3,  random_state = 0, learning_rate = "auto", metric = 'precomputed')
trans_data = tsne.fit_transform(jac_sim).T

fig = plt.figure(figsize = (12,12))
ax = plt.axes(projection = '3d')

ax.scatter(trans_data[0], trans_data[1], trans_data[2], c = y.ravel(), 
           cmap =  ListedColormap(['black','red']), s = 5)
ax.text3D(0, 0, 15, "Real Problem?", color = 'blue', weight = 'bold', fontsize = 'xx-large', fontstyle = 'italic')

a = Arrow3D([2, 1.5], [3, 1.5], [13, 3], mutation_scale = 20,
            lw = 2, arrowstyle = "-|>", color = "blue")
ax.add_artist(a)

plt.title('Traces')

In [ ]:
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d

In [ ]:
 class Arrow3D(FancyArrowPatch):

    def __init__(self, xs, ys, zs, *args, **kwargs):
        FancyArrowPatch.__init__(self, (0, 0), (0, 0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def draw(self, renderer):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, renderer.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        FancyArrowPatch.draw(self, renderer)